# 00 — MASTER: Start Here

This accelerator has grown a lot of moving parts (10 notebooks, 14 script modules, 4 test files, several company graphs, CPU **and** GPU solvers). **This notebook is the map + the smoke test.** Run it top to bottom and by the end you'll know: what everything is, whether your environment is healthy, that the core engines actually work, and exactly which notebook to open for what you want to do.

Nothing here is destructive: it only generates in-memory data and solves small models. It writes no files and touches no Databricks jobs.

| Section | What it does | Needs |
|---|---|---|
| 1. The map | One-paragraph tour of every notebook & module | reading only |
| 2. Environment check | Confirms Python + the key libraries import | `requirements.txt` |
| 3. Smoke test | Generates a network and runs TTR / TTS / FJSP-derive / CVRPTW-derive / MEIO end-to-end, CPU-only | `requirements.txt` |
| 4. Run the tests | Executes the pytest suite (the real regression guard) | `pytest` |
| 5. Where do I go next? | A decision guide → the right notebook | reading only |

## Cluster configuration

Everything in **this** notebook runs on a plain single-node CPU cluster (the same one `05`–`08` use):
- **Databricks Runtime:** 17.3 LTS ML
- **Single Node** — Azure `Standard_DS4_v2` / AWS `m5d.2xlarge`

Only the GPU sections of notebook `09` need a GPU cluster; this master driver never does.

In [ ]:
%pip install -r ./requirements.txt --quiet
dbutils.library.restartPython()

## 1. The map — what is everything?

### Notebooks (run in order, or jump to what you need)

| Notebook | Purpose | Compute |
|---|---|---|
| `00_introduction` | The methodology (TTR/TTS digital-twin stress testing) in prose | read |
| `01_operational_data` | Generate the original **toy** synthetic network (random params) | CPU |
| `02_stress_testing (small network)` | Single-node stress test on the small toy network | CPU |
| `03_stress_testing (large network)` | Ray-distributed stress test on the large toy network | CPU (Ray) |
| `04_appendix` | Extra background / derivations | read |
| `05_realistic_operational_data` | Generate the **realistic** networks (simple / medium / complex: Nvidia / Apple / MPS / **apple_real**), + planet-scale | CPU |
| `06_realistic_stress_testing (simple and medium)` | Named disruption scenarios on the simple/medium networks | CPU |
| `07_realistic_stress_testing (complex network)` | Ray-distributed scenarios on the Nvidia/Apple complex networks | CPU (Ray) |
| `08_multi_period_planning` | Time-phased multi-period planning + network decomposition (Adexa-inspired) | CPU |
| `09_mps_cuopt_pipeline` | **FJSP / CVRPTW / MEIO** on NVIDIA cuOpt (GPU) + Pyomo (CPU); `company` widget = mps / apple_real | CPU + optional GPU |

### Script modules (`scripts/`) — the reusable engines

- **`utils.py`** — the LP engine: `build_and_solve_ttr` / `_tts` + 6 more objectives + a resilience metric (9 total). *Unmodified across all the realistic work.*
- **`realistic_topologies.py`** — generates the realistic networks; `generate_complex_network(company)` / `_at_scale(company, scale="planet")`.
- **`company_profiles.py`** — the real/illustrative company anchors (nvidia, apple, mps, apple_real).
- **`apple_supplier_list.py`** — parses Apple's **published** supplier list → the `apple_real` anchors.
- **`scenario_calibration.py`** — turns topology + criticality into correlated params (`calibrate_network`, `calibrate_cost_fields`).
- **`disruption_scenarios.py`** — named single-supplier / regional / material-wide failure scenarios.
- **`multi_period_planning.py`**, **`network_aggregation.py`** — the notebook-08 engines.
- **`mps_derivation.py`** — turns a generated network into FJSP/CVRPTW/MEIO inputs; company-parameterized via `DerivationConfig` (`MPS_CONFIG` / `APPLE_REAL_CONFIG`).
- **`fjsp_cuopt.py`**, **`cvrptw_cuopt.py`** — GPU (NVIDIA cuOpt) models; **`meio_pyomo.py`** — CPU (Pyomo/HiGHS) model.
- **`dataset_io.py`** — explode/reconstruct a dataset to/from tables.

### Deep-dive docs (in `docs/`)
`optimization-objectives.md` · `disruption-scenarios.md` · `multi-period-and-decomposition.md` · `mps-cuopt-pipeline.md`

## 2. Environment check

Confirms the libraries the CPU path needs are importable. cuOpt/cuDF are **expected to be missing** on a CPU cluster — they're only for notebook `09`'s GPU sections.

In [ ]:
import importlib
import sys

print(f"Python {sys.version.split()[0]}\n")

REQUIRED = ["pyomo", "highspy", "pandas", "numpy"]
OPTIONAL = ["ray", "networkx"]
GPU_ONLY = ["cuopt", "cudf"]

def _check(mods, label):
    for m in mods:
        try:
            importlib.import_module(m)
            print(f"  [ok]   {m}")
        except Exception as e:
            print(f"  [MISS] {m}  ({label})  -> {type(e).__name__}")

print("Required (CPU path):")
_check(REQUIRED, "install requirements.txt")
print("\nOptional (some notebooks):")
_check(OPTIONAL, "only 03/07 need ray; some viz needs networkx")
print("\nGPU-only (notebook 09 GPU sections — expected missing on CPU):")
_check(GPU_ONLY, "install requirements-gpu.txt on a GPU cluster")

# The repo's own modules must import (confirms scripts/ is on the path).
print("\nRepo modules:")
_check(
    [
        "scripts.utils",
        "scripts.realistic_topologies",
        "scripts.mps_derivation",
        "scripts.meio_pyomo",
        "scripts.apple_supplier_list",
    ],
    "run this notebook from the repo root",
)

## 3. Smoke test — prove the core engines work (CPU-only, ~30–60s)

This is the single "is everything wired up?" check. For **both** company graphs (`mps` and `apple_real`) it:
1. generates a realistic network and calibrates its parameters,
2. runs the **LP engine** (`build_and_solve_ttr`) — the core of `02`–`08`,
3. **derives** the FJSP machines, CVRPTW depots, and MEIO network (the notebook-09 layer),
4. **solves MEIO** on CPU (Pyomo/HiGHS) and bounds its approximation error.

If this cell prints `ALL SMOKE TESTS PASSED`, the whole CPU pipeline is healthy and you can confidently open any notebook. The cuOpt GPU solves (FJSP/CVRPTW) are the only thing NOT exercised here — they need a GPU cluster (see §5).

In [ ]:
import random
import time

import scripts.realistic_topologies as rt
import scripts.scenario_calibration as sc
import scripts.mps_derivation as md
import scripts.meio_pyomo as meio
import scripts.utils as utils

DERIVATION_CONFIGS = {"mps": md.MPS_CONFIG, "apple_real": md.APPLE_REAL_CONFIG}

def _dataset(company):
    d = rt.generate_complex_network(company)
    fields = sc.calibrate_cost_fields(
        random.Random(7),
        d["tier1"], d["tier2"], d["tier3"],
        d["material_types"], d["supplier_material_type"],
        d["edges"], d.get("criticality"), company,
        region=d.get("region"),
    )
    return {**d, **fields}

all_ok = True
for company in ["mps", "apple_real"]:
    t0 = time.time()
    cfg = DERIVATION_CONFIGS[company]
    d = _dataset(company)
    n_nodes = len(d["tier1"]) + len(d["tier2"]) + len(d["tier3"])

    # (2) LP engine — the core stress-test solver used by 02-08
    ttr = utils.build_and_solve_ttr(d, [], 10)
    lp_ok = str(ttr.iloc[0]["termination_condition"]).lower() == "optimal"

    # (3) derive the notebook-09 problems
    machines = md.select_backend_machines(d, cfg)
    depots = md.select_cvrptw_depots(d, cfg)
    jobs = md.derive_fjsp_jobs(d, machines, config=cfg)
    pt = md.derive_fjsp_processing_times(jobs, machines, d, config=cfg)
    derive_ok = bool(machines and depots and jobs) and all(
        any((j["job_id"], op, m["machine_id"]) in pt for m in machines)
        for j in jobs for op in j["operations"]
    )

    # (4) MEIO solve on CPU + approximation-error bound
    net = md.derive_meio_network(d, config=cfg)
    res = meio.solve_meio(meio.build_meio_model(net))
    true = meio.true_safety_stock_cost(net, res["per_node"])
    rel = abs(res["total_safety_stock_cost"] - true) / max(1e-9, true)
    meio_ok = res["termination_condition"].lower() == "optimal" and rel < 0.05

    ok = lp_ok and derive_ok and meio_ok
    all_ok &= ok
    flag = "PASS" if ok else "FAIL"
    print(
        f"[{flag}] {company:10s} nodes={n_nodes:5d} | LP={'ok' if lp_ok else 'X'} | "
        f"machines={len(machines)} depots={len(depots)} jobs={len(jobs)} derive={'ok' if derive_ok else 'X'} | "
        f"MEIO={res['termination_condition']} err={rel:.2%} | {time.time()-t0:.1f}s"
    )

print("\n" + ("ALL SMOKE TESTS PASSED ✅" if all_ok else "SMOKE TEST FAILURE ❌ — see the FAIL row above"))

### Optional: confirm the planet-scale path (~76k nodes, adds ~30–60s)

Generates a planet-scale graph and runs the **subsampled** MEIO (the full 76k-node solve would take ~23 min; the pipeline samples down to keep it interactive). Skip if you only care about complex scale.

In [ ]:
run_planet_smoke = True  # set False to skip

if run_planet_smoke:
    for company in ["mps", "apple_real"]:
        cfg = DERIVATION_CONFIGS[company]
        t0 = time.time()
        dp = rt.generate_complex_network_at_scale(company, scale="planet")
        fields = sc.calibrate_cost_fields(
            random.Random(7), dp["tier1"], dp["tier2"], dp["tier3"],
            dp["material_types"], dp["supplier_material_type"], dp["edges"],
            dp.get("criticality"), company, region=dp.get("region"),
        )
        dp = {**dp, **fields}
        n = len(dp["tier1"]) + len(dp["tier2"]) + len(dp["tier3"])
        machines = md.select_backend_machines(dp, cfg)  # must be scale-invariant
        net = md.derive_meio_network(dp, max_nodes=2500, config=cfg)
        res = meio.solve_meio(meio.build_meio_model(net))
        print(
            f"[{company}] planet nodes={n} machines={len(machines)} (scale-invariant) | "
            f"MEIO sampled to {len(net['nodes'])} nodes={res['termination_condition']} | {time.time()-t0:.1f}s"
        )
else:
    print("planet smoke skipped")

## 4. Run the test suite

The pytest suite is the real regression guard (schema validity, LP solvability, derivation correctness, scale-invariance, MEIO error bounds, the parser). GPU-dependent tests **skip** automatically without cuOpt installed. Runs from the repo root.

In [ ]:
import subprocess
import sys

# Use the running kernel's interpreter (robust on Databricks, where a bare
# "python" may not be on PATH). Falls back to "-m pytest" so the repo's
# pyproject pytest config is picked up. Run from the repo root.
proc = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-q"],
    capture_output=True, text=True,
)
print(proc.stdout[-4000:])
if proc.stderr:
    print("--- stderr (tail) ---")
    print(proc.stderr[-1500:])
print("\nexit code:", proc.returncode, "(0 = all passed / skipped)")

## 5. Where do I go next?

**Pick by what you want to do:**

| I want to… | Open… |
|---|---|
| Understand the TTR/TTS methodology from scratch | `00_introduction`, then `01`→`02` |
| See stress testing on a **realistic** supply chain | `05_realistic_operational_data` (generate) → `06` (simple/medium) → `07` (Nvidia/Apple, Ray) |
| Explore **time-phased / multi-period** planning | `08_multi_period_planning` |
| Run **scheduling (FJSP) / routing (CVRPTW) / inventory (MEIO)** with cuOpt | `09_mps_cuopt_pipeline` — set `company` = `mps` or `apple_real` |
| Use the **real Apple supplier-list** graph | `09` with `company=apple_real` (data comes from `05`) |
| Push to **planet scale** (~76k nodes) | `05` planet section (generate) → `09` with `scale=planet` |
| Actually run cuOpt on **GPU** | `09` on a Databricks **serverless GPU** cluster with `run_gpu=yes` (see its support caveat) |

**Recommended first run (all CPU):**
1. This notebook top-to-bottom — confirm `ALL SMOKE TESTS PASSED` + green pytest.
2. `05_realistic_operational_data` with `regenerate_data=yes` — writes all the datasets to the volume.
3. `09_mps_cuopt_pipeline` with `company=mps`, `scale=complex`, `run_gpu=no` — the full derive + CPU MEIO path.
4. Then flip `09` to `company=apple_real` and/or `scale=planet` to see the same engine on the real Apple graph / at scale.

**Only when you have a GPU cluster:** `09` with `run_gpu=yes` — this is the only path that exercises the actual NVIDIA cuOpt FJSP/CVRPTW GPU solves (technically feasible but **not** an officially supported Databricks configuration; see `docs/mps-cuopt-pipeline.md`).

&copy; 2025 Databricks, Inc. All rights reserved. The source in this notebook is provided subject to the Databricks License [https://databricks.com/db-license-source]. All included or referenced third party libraries are subject to the licenses set forth below.

| library | description | license | source |
|---|---|---|---|
| pyomo | Algebraic modeling language for optimization | BSD-3 | https://pypi.org/project/pyomo/ |
| highspy | Linear optimization solver (HiGHS) | MIT | https://pypi.org/project/highspy/ |
| ray | Framework for scaling AI/Python applications | Apache 2.0 | https://github.com/ray-project/ray |